In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
# 1. RUTAS DE TUS ARCHIVOS EN DRIVE
ruta_tsv = '/content/drive/MyDrive/proyecto practica/Base de datos/drugbank_final.tsv'
ruta_excel = '/content/drive/MyDrive/proyecto practica/Base de datos/ddiData.xlsx'

print("Cargando las bases de datos... (esto puede tomar unos segundos para el archivo de 400MB)")
# Cargamos el TSV
df_tsv = pd.read_csv(ruta_tsv, sep='\t', low_memory=False)
# Cargamos el Excel
df_excel = pd.read_excel(ruta_excel)

# 2. CONFIGURACIÓN EXACTA DE TUS COLUMNAS
col_nombre_tsv = 'name'
col_desc_tsv = 'description'

col_nombre_excel = 'drug_name'
col_desc_excel = 'text'

# 3. EL MOTOR DE BÚSQUEDA COMPARATIVA
def comparar_farmacos(lista_de_busqueda):
    for farmaco in lista_de_busqueda:
        print(f"\n" + "="*80)
        print(f"💊 COMPARANDO: {farmaco.upper()}")
        print("="*80)

        # --- BÚSQUEDA EN EL TSV (DrugBank) ---
        res_tsv = df_tsv[df_tsv[col_nombre_tsv].astype(str).str.contains(farmaco, case=False, na=False)]

        print("\n📌 SEGÚN DRUGBANK (TSV):")
        if not res_tsv.empty:
            # Extraemos la descripción del primer resultado encontrado
            desc_tsv = res_tsv.iloc[0][col_desc_tsv]
            print(desc_tsv)
        else:
            print(f"❌ No se encontró '{farmaco}' en el archivo TSV.")

        # --- BÚSQUEDA EN TU EXCEL ---
        res_excel = df_excel[df_excel[col_nombre_excel].astype(str).str.contains(farmaco, case=False, na=False)]

        print("\n📌 SEGÚN TU EXCEL:")
        if not res_excel.empty:
            # Extraemos la descripción del primer resultado encontrado
            desc_excel = res_excel.iloc[0][col_desc_excel]
            print(desc_excel)
        else:
            print(f"❌ No se encontró '{farmaco}' en el archivo Excel.")

        print("\n")


# 4. LISTA DE FÁRMACOS A COMPARAR
# Escribe entre comillas los fármacos que quieres buscar hoy, separados por comas.
farmacos_a_comparar = ["Lepirudin", "Abacavir", "Aspirin"]

# Ejecutamos la función
comparar_farmacos(farmacos_a_comparar)

Cargando las bases de datos... (esto puede tomar unos segundos para el archivo de 400MB)

💊 COMPARANDO: LEPIRUDIN

📌 SEGÚN DRUGBANK (TSV):
Lepirudin is a recombinant hirudin formed by 65 amino acids that acts as a highly specific and direct thrombin inhibitor.[L41539,L41569] Natural hirudin is an endogenous anticoagulant found in _Hirudo medicinalis_ leeches.[L41539] Lepirudin is produced in yeast cells and is identical to natural hirudin except for the absence of sulfate on the tyrosine residue at position 63 and the substitution of leucine for isoleucine at position 1 (N-terminal end).[A246609] Lepirudin is used as an anticoagulant in patients with heparin-induced thrombocytopenia (HIT), an immune reaction associated with a high risk of thromboembolic complications.[A3, L41539] HIT is caused by the expression of immunoglobulin G (IgG) antibodies that bind to the complex formed by heparin and platelet factor 4. This activates endothelial cells and platelets and enhances the formation 

In [ ]:
import pandas as pd
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import os
from google.colab import drive

# 1. Asegurar que Drive esté montado
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# --- CONFIGURACIÓN DE RUTAS ---
ruta_tsv = '/content/drive/MyDrive/proyecto practica/Base de datos/drugbank_final.tsv'
ruta_excel_destino = '/content/drive/MyDrive/proyecto practica/Base de datos/ddiData.xlsx'
nombre_hoja = 'Analisis_Combinado'

if not os.path.exists(ruta_tsv):
    print(f"❌ ERROR: No se encuentra el archivo TSV en: {ruta_tsv}")
else:
    print("⏳ Cargando datos y procesando información...")

    # 1. Carga de archivos
    df_tsv = pd.read_csv(ruta_tsv, sep='\t', low_memory=False)
    df_excel = pd.read_excel(ruta_excel_destino)

    # 2. Identificación de columnas
    col_nombre_tsv = 'name'
    col_id_tsv = 'drugbank-id' if 'drugbank-id' in df_tsv.columns else df_tsv.columns[0]
    col_interacciones_tsv = df_tsv.columns[13]  # Columna 14 (drug-interactions)

    col_nombre_excel = 'drug_name'
    col_desc_excel = 'text'

    # 3. Limpieza y Cruce
    df_tsv_subset = df_tsv[[col_id_tsv, col_nombre_tsv, col_interacciones_tsv]].copy()
    df_tsv_subset['nombre_norm'] = df_tsv_subset[col_nombre_tsv].astype(str).str.strip().str.lower()
    df_tsv_subset = df_tsv_subset.drop_duplicates(subset=['nombre_norm'])

    df_excel['nombre_norm'] = df_excel[col_nombre_excel].astype(str).str.strip().str.lower()

    df_unificado = pd.merge(df_excel, df_tsv_subset, on='nombre_norm', how='left')

    # 4. Generar formato de celda pedido
    def formato_final(row):
        info_ex = str(row[col_desc_excel]).strip() if pd.notna(row[col_desc_excel]) else 'Sin información en Excel'
        info_ts = str(row[col_interacciones_tsv]).strip() if pd.notna(row[col_interacciones_tsv]) else 'Sin interacciones en TSV'
        return f"excel: {info_ex}\n\ntsv: {info_ts}"

    df_unificado['Información Combinada'] = df_unificado.apply(formato_final, axis=1)

    # 5. Selección y renombre de columnas
    df_resultado = df_unificado[[col_id_tsv, col_nombre_excel, 'Información Combinada']].rename(
        columns={col_id_tsv: 'ID DrugBank', col_nombre_excel: 'Nombre Droga'}
    )

    # 6. Guardar en Excel
    print(f"Guardando datos en la hoja '{nombre_hoja}'...")
    with pd.ExcelWriter(ruta_excel_destino, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        df_resultado.to_excel(writer, sheet_name=nombre_hoja, index=False)

    # 7. Aplicar estilos limpios (sin generar conflicto con table1.xml)
    print("Aplicando estilos visuales en verde y bordes...")
    wb = openpyxl.load_workbook(ruta_excel_destino)
    ws = wb[nombre_hoja]

    # Colores ARGB compatibles con Excel de escritorio y Google Sheets
    fill_header = PatternFill(fill_type="solid", start_color="FF2E7D32", end_color="FF2E7D32")
    font_header = Font(name="Arial", size=10, bold=True, color="FFFFFFFF")

    fill_par = PatternFill(fill_type="solid", start_color="FFE8F5E9", end_color="FFE8F5E9")
    fill_impar = PatternFill(fill_type="solid", start_color="FFFFFFFF", end_color="FFFFFFFF")
    font_body = Font(name="Arial", size=10, color="FF000000")

    thin_border = Border(
        left=Side(style='thin', color='FFC8E6C9'),
        right=Side(style='thin', color='FFC8E6C9'),
        top=Side(style='thin', color='FFC8E6C9'),
        bottom=Side(style='thin', color='FFC8E6C9')
    )

    # Formato encabezado (Fila 1)
    ws.row_dimensions[1].height = 28
    for col in range(1, ws.max_column + 1):
        cell = ws.cell(row=1, column=col)
        cell.fill = fill_header
        cell.font = font_header
        cell.alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)

    # Formato filas alternadas y saltos de línea
    for row in range(2, ws.max_row + 1):
        current_fill = fill_par if row % 2 == 0 else fill_impar
        for col in range(1, ws.max_column + 1):
            cell = ws.cell(row=row, column=col)
            cell.fill = current_fill
            cell.font = font_body
            cell.border = thin_border
            cell.alignment = Alignment(horizontal="left", vertical="top", wrap_text=True)

    # Anchos de columna
    ws.column_dimensions['A'].width = 18
    ws.column_dimensions['B'].width = 25
    ws.column_dimensions['C'].width = 110

    # Guardar cambios
    wb.save(ruta_excel_destino)
    os.sync()

    print(f"\n✅ ¡Archivo generado sin errores de recuperación! Hoja '{nombre_hoja}' lista para abrirse en el PC.")

⏳ Cargando datos y procesando información...
Guardando datos en la hoja 'Analisis_Combinado'...
Aplicando estilos visuales en verde y bordes...

✅ ¡Archivo generado sin errores de recuperación! Hoja 'Analisis_Combinado' lista para abrirse en el PC.
